In [4]:
import numpy as np
import pandas as pd

### 1. Environment Setup & Data Loading

In [2]:
df = pd.read_csv(filepath_or_buffer="All_State_NCRB_Crime_Data_2014.csv")
df.head()

,States/UTs,District,Year,Murder,Attempt to commit Murder,Culpable Homicide not amounting to Murder,Attempt to commit Culpable Homicide,Rape,Custodial Rape,Custodial_Gang Rape,...,Offences promoting enmity between different groups,Promoting enmity between different groups,"Imputation, assertions prejudicial to national integration",Extortion,Disclosure of Identity of Victims,Incidence of Rash Driving,HumanTrafficking,Unnatural Offence,Other IPC crimes,Total Cognizable IPC crimes
0,Andhra Pradesh,Anantapur,2014,134,171,8,0,35,0,0,...,0,0,0,0,0,1038,0,0,3800,8376
1,Andhra Pradesh,Chittoor,2014,84,170,2,0,32,0,0,...,0,0,0,19,0,249,0,0,2567,5374
2,Andhra Pradesh,Cuddapah,2014,80,162,1,0,28,0,0,...,0,0,0,0,0,948,0,0,2604,5803
3,Andhra Pradesh,East Godavari,2014,64,84,2,0,85,0,0,...,0,0,0,32,0,39,0,0,3791,7630
4,Andhra Pradesh,Guntakal Railway,2014,14,4,0,0,0,0,0,...,0,0,0,0,0,1,0,0,37,490


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 838 entries, 0 to 837
Data columns (total 91 columns):
 #   Column                                                            Non-Null Count  Dtype
---  ------                                                            --------------  -----
 0   States/UTs                                                        838 non-null    str  
 1   District                                                          838 non-null    str  
 2   Year                                                              838 non-null    int64
 3   Murder                                                            838 non-null    int64
 4   Attempt to commit Murder                                          838 non-null    int64
 5   Culpable Homicide not amounting to Murder                         838 non-null    int64
 6   Attempt to commit Culpable Homicide                               838 non-null    int64
 7   Rape                                                            

In [7]:
NUMERICAL_COLS = df.select_dtypes(include=np.number).columns
CATEGORICAL_COLS = df.select_dtypes(include="str").columns

In [56]:
len((df["States/UTs"]).value_counts())

36

### 2. Separation of Aggregates vs. District Granularity

In [11]:
df.head()

,States/UTs,District,Year,Murder,Attempt to commit Murder,Culpable Homicide not amounting to Murder,Attempt to commit Culpable Homicide,Rape,Custodial Rape,Custodial_Gang Rape,...,Offences promoting enmity between different groups,Promoting enmity between different groups,"Imputation, assertions prejudicial to national integration",Extortion,Disclosure of Identity of Victims,Incidence of Rash Driving,HumanTrafficking,Unnatural Offence,Other IPC crimes,Total Cognizable IPC crimes
0,Andhra Pradesh,Anantapur,2014,134,171,8,0,35,0,0,...,0,0,0,0,0,1038,0,0,3800,8376
1,Andhra Pradesh,Chittoor,2014,84,170,2,0,32,0,0,...,0,0,0,19,0,249,0,0,2567,5374
2,Andhra Pradesh,Cuddapah,2014,80,162,1,0,28,0,0,...,0,0,0,0,0,948,0,0,2604,5803
3,Andhra Pradesh,East Godavari,2014,64,84,2,0,85,0,0,...,0,0,0,32,0,39,0,0,3791,7630
4,Andhra Pradesh,Guntakal Railway,2014,14,4,0,0,0,0,0,...,0,0,0,0,0,1,0,0,37,490


In [18]:
df_district = df[df["District"] != "Total"]
df_state_totals = df[df["District"] == "Total"]

if (
    df_district["Total Cognizable IPC crimes"].sum()
    == df_state_totals["Total Cognizable IPC crimes"].sum()
):
    print("No double-counting included")
else:
    print("There is some double-counting")


No double-counting included


### 3. Identifying Special Administrative Units (Railway & Crime Branch Districts)

In [24]:
non_standard_places = ["Railway", "Crime Branch", "CID"]

pattern = "|".join(non_standard_places)

non_standard_districts = df[
    df["District"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True,
    )
]

non_standard_districts.head()

,States/UTs,District,Year,Murder,Attempt to commit Murder,Culpable Homicide not amounting to Murder,Attempt to commit Culpable Homicide,Rape,Custodial Rape,Custodial_Gang Rape,...,Offences promoting enmity between different groups,Promoting enmity between different groups,"Imputation, assertions prejudicial to national integration",Extortion,Disclosure of Identity of Victims,Incidence of Rash Driving,HumanTrafficking,Unnatural Offence,Other IPC crimes,Total Cognizable IPC crimes
4,Andhra Pradesh,Guntakal Railway,2014,14,4,0,0,0,0,0,...,0,0,0,0,0,1,0,0,37,490
15,Andhra Pradesh,Vijayawada Railway,2014,1,1,1,0,0,0,0,...,0,0,0,1,0,0,0,0,59,1223
23,Arunachal Pradesh,Crime Branch,2014,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
84,Bihar,Jamalpur Railway,2014,9,1,0,0,0,0,0,...,0,0,0,2,0,0,0,0,72,298
88,Bihar,Katihar Railway,2014,10,9,0,0,0,0,0,...,0,0,0,0,0,0,3,0,61,313


In [28]:
avg_murder_df = non_standard_districts.groupby(["States/UTs", "District"]).agg(
    avg_murder=("Murder", "mean"),
)

avg_murder_df.head()

avg_murder
States/UTs        District                      
Andhra Pradesh    Guntakal Railway          14.0
                  Vijayawada Railway         1.0
Arunachal Pradesh Crime Branch               0.0
Bihar             Jamalpur Railway           9.0
                  Katihar Railway           10.0

### 4. State-Wise Ranking of Cognizable Crimes

In [47]:
state_wise_crimes = (
    df_state_totals.groupby("States/UTs")
    .agg(
        total_crime=("Total Cognizable IPC crimes", "sum"),
    )
    .reset_index()
)

TOTAL_CRIME_ACROSS_STATES = state_wise_crimes["total_crime"].sum()

state_wise_crimes["distribution_perc"] = np.round(
    (state_wise_crimes["total_crime"] / TOTAL_CRIME_ACROSS_STATES) * 100, 2
)

state_wise_crimes = state_wise_crimes.sort_values(
    by="distribution_perc", ascending=False
).reset_index()

top5 = state_wise_crimes.head(5)
bottom5 = state_wise_crimes.tail(5)

In [48]:
top5

,index,States/UTs,total_crime,distribution_perc
0,19,Madhya Pradesh,272423,9.53
1,20,Maharashtra,249834,8.74
2,33,Uttar Pradesh,240475,8.41
3,28,Rajasthan,210418,7.36
4,17,Kerala,206789,7.23


In [49]:
bottom5

,index,States/UTs,total_crime,distribution_perc
31,29,Sikkim,1065,0.04
32,0,A&N Islands,746,0.03
33,8,Daman & Diu,233,0.01
34,7,D&N Haveli,277,0.01
35,18,Lakshadweep,81,0.00


### 5. Analysis of Violent Crimes Against Person

In [50]:
df_state_totals.head()

,States/UTs,District,Year,Murder,Attempt to commit Murder,Culpable Homicide not amounting to Murder,Attempt to commit Culpable Homicide,Rape,Custodial Rape,Custodial_Gang Rape,...,Offences promoting enmity between different groups,Promoting enmity between different groups,"Imputation, assertions prejudicial to national integration",Extortion,Disclosure of Identity of Victims,Incidence of Rash Driving,HumanTrafficking,Unnatural Offence,Other IPC crimes,Total Cognizable IPC crimes
20,Andhra Pradesh,Total,2014,1175,1540,52,1,961,0,0,...,21,21,0,276,0,14653,2,1,44187,114604
40,Arunachal Pradesh,Total,2014,86,48,4,0,83,4,0,...,0,0,0,121,0,102,0,0,720,2843
69,Assam,Total,2014,1451,1142,57,14,1980,0,0,...,0,0,0,1226,0,3202,68,0,29212,94337
116,Bihar,Total,2014,3403,4379,201,463,1127,0,0,...,0,0,0,972,0,3485,44,5,75799,177595
145,Chhattisgarh,Total,2014,998,716,29,0,1436,0,0,...,3,3,0,74,0,7759,43,22,21897,58200


In [62]:
violent_crimes = df_state_totals.copy(deep=True)

target_cols = [
    "Murder",
    "Attempt to commit Murder",
    "Culpable Homicide not amounting to Murder",
    "Attempt to commit Culpable Homicide",
]

violent_crimes["Violent_Crimes_Person"] = df[target_cols].sum(axis=1)

violent_crimes = violent_crimes[["States/UTs", "Violent_Crimes_Person"]].reset_index()
violent_crimes.head(10)

,index,States/UTs,Violent_Crimes_Person
0,20,Andhra Pradesh,2768
1,40,Arunachal Pradesh,138
2,69,Assam,2664
3,116,Bihar,8446
4,145,Chhattisgarh,1743
5,149,Goa,82
6,191,Gujarat,1940
7,216,Haryana,1963
8,232,Himachal Pradesh,202
9,263,Jammu & Kashmir,660
